In [ ]:
# Adapting Pretrained Models: Prompting, PEFT, Quantization
# Generated from the canonical HTML manuscript. Run this cell first.
# Source: https://github.com/Shakeri-Lab/dl-book/blob/c058d1f401fd0ead3ae59a2a8730f95489a2d9aa/chapters/part5/17-peft-quantization.qmd

from importlib.metadata import PackageNotFoundError, version as package_version
import hashlib as _bootstrap_hashlib
import os as _bootstrap_os
from pathlib import Path as _BootstrapPath
import subprocess as _bootstrap_subprocess
import sys as _bootstrap_sys
import urllib.request as _bootstrap_urlrequest

_BOOK_REVISION = 'c058d1f401fd0ead3ae59a2a8730f95489a2d9aa'
_PINNED_REQUIREMENTS = [
    "torch==2.12.1",
    "torchvision==0.27.1",
    "numpy==2.5.1",
    "matplotlib==3.11.1"
]
_BOOK_ASSETS = []

def _installed_requirement(requirement: str) -> bool:
    name, expected = requirement.split('==', 1)
    try:
        return package_version(name) == expected
    except PackageNotFoundError:
        return False

_missing_requirements = [
    item for item in _PINNED_REQUIREMENTS if not _installed_requirement(item)
]
if _missing_requirements:
    _bootstrap_install = _bootstrap_subprocess.run(
        [_bootstrap_sys.executable, '-m', 'pip', 'install', '--quiet',
         *_missing_requirements],
        check=False, capture_output=True, text=True,
    )
    if _bootstrap_install.returncode != 0:
        raise RuntimeError(_bootstrap_install.stdout + _bootstrap_install.stderr)

_bootstrap_base = _BootstrapPath(
    _bootstrap_os.environ.get(
        'DLBOOK_NOTEBOOK_ROOT',
        '/content' if _BootstrapPath('/content').is_dir()
        else str(_BootstrapPath.home() / '.cache'),
    )
)
_BOOK_ROOT = _bootstrap_base / f'dl-book-{_BOOK_REVISION[:12]}'
_RAW_ROOT = 'https://raw.githubusercontent.com/Shakeri-Lab/dl-book/' + _BOOK_REVISION + '/'
for _record in _BOOK_ASSETS:
    _destination = _BOOK_ROOT / _record['path']
    _destination.parent.mkdir(parents=True, exist_ok=True)
    _valid = (
        _destination.is_file()
        and _bootstrap_hashlib.sha256(_destination.read_bytes()).hexdigest()
        == _record['sha256']
    )
    if not _valid:
        _temporary = _destination.with_suffix(_destination.suffix + '.part')
        _bootstrap_urlrequest.urlretrieve(_RAW_ROOT + _record['path'], _temporary)
        _digest = _bootstrap_hashlib.sha256(_temporary.read_bytes()).hexdigest()
        if _digest != _record['sha256']:
            _temporary.unlink(missing_ok=True)
            raise RuntimeError(f"Checksum mismatch for {_record['path']}")
        _temporary.replace(_destination)

(_BOOK_ROOT / 'chapters/part5').mkdir(parents=True, exist_ok=True)
_bootstrap_sys.path.insert(0, str(_BOOK_ROOT / 'code'))
_bootstrap_os.chdir(_BOOK_ROOT / 'chapters/part5')

# Hidden manuscript support required by later learner-visible cells.
# Plot-only harnesses are not exported.
import torch
from torch import nn

import math

import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F

torch.set_num_threads(6)
torch.manual_seed(6050)

navy, orange, green, wine = "#232D4B", "#E57200", "#2E7D32", "#722F37"

assert _BOOK_ROOT.is_dir()

**Plan**

1. Prepare the inputs and fixed settings for the example.
2. Define the reusable helpers: `sample_episode_bank`, `render_episodes`, and `ContextLearner`.
3. Define the reusable helpers: `evaluate_context` and `run_context_seed`.
4. Meta-train a tiny causal Transformer, then audit frozen-weight in-context adaptation over five seeds.

In [ ]:
# [1]
PAD, BOS, QUERY = 0, 1, 2
GROUP0, LABEL0 = 3, 7
VOCAB, SEQUENCE_LENGTH, N_CLASSES = 11, 11, 4


# [2]
def sample_episode_bank(
    batch_size: int, generator: torch.Generator
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    mapping = torch.rand(batch_size, N_CLASSES, generator=generator).argsort(dim=1)
    demo_groups = torch.rand(
        batch_size, N_CLASSES, generator=generator
    ).argsort(dim=1)
    query_groups = torch.randint(
        N_CLASSES, (batch_size,), generator=generator
    )
    return mapping, demo_groups, query_groups


def render_episodes(
    bank: tuple[torch.Tensor, torch.Tensor, torch.Tensor], k: int
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    mapping, demo_groups, query_groups = bank
    tokens = torch.full((len(mapping), SEQUENCE_LENGTH), PAD, dtype=torch.long)
    tokens[:, 0] = BOS
    for slot in range(k):
        group = demo_groups[:, slot]
        label = mapping.gather(1, group[:, None]).squeeze(1)
        tokens[:, 1 + 2 * slot] = GROUP0 + group
        tokens[:, 2 + 2 * slot] = LABEL0 + label
    tokens[:, 9] = QUERY
    tokens[:, 10] = GROUP0 + query_groups
    targets = mapping.gather(1, query_groups[:, None]).squeeze(1)
    covered = (
        (demo_groups[:, :k] == query_groups[:, None]).any(dim=1)
        if k else torch.zeros(len(mapping), dtype=torch.bool)
    )
    return tokens, targets, covered


class ContextLearner(nn.Module):
    def __init__(self, width: int = 48, heads: int = 4, layers: int = 2) -> None:
        super().__init__()
        self.token = nn.Embedding(VOCAB, width, padding_idx=PAD)
        self.position = nn.Embedding(SEQUENCE_LENGTH, width)
        layer = nn.TransformerEncoderLayer(
            d_model=width, nhead=heads, dim_feedforward=2 * width,
            dropout=0.0, batch_first=True, norm_first=True, activation="gelu",
        )
        self.encoder = nn.TransformerEncoder(
            layer, layers, norm=nn.LayerNorm(width), enable_nested_tensor=False
        )
        self.head = nn.Linear(width, N_CLASSES)
        self.register_buffer(
            "causal",
            torch.triu(torch.ones(SEQUENCE_LENGTH, SEQUENCE_LENGTH, dtype=torch.bool),
                       diagonal=1),
        )

    def forward(self, tokens: torch.Tensor) -> torch.Tensor:
        positions = torch.arange(SEQUENCE_LENGTH, device=tokens.device)
        hidden = self.token(tokens) + self.position(positions)[None]
        hidden = self.encoder(
            hidden, mask=self.causal, src_key_padding_mask=tokens.eq(PAD)
        )
        return self.head(hidden[:, -1])


# [3]
@torch.no_grad()
def evaluate_context(
    model: nn.Module, seed: int, n: int = 8192
) -> tuple[list[dict], bool]:
    model.eval()
    before = [parameter.detach().clone() for parameter in model.parameters()]
    bank = sample_episode_bank(n, torch.Generator().manual_seed(seed + 10_000))
    rows = []
    for k in range(5):
        tokens, targets, covered = render_episodes(bank, k)
        correct = model(tokens).argmax(1).eq(targets)
        rows.append({
            "k": k,
            "accuracy": correct.float().mean().item(),
            "covered": correct[covered].float().mean().item()
                       if covered.any() else float("nan"),
            "uncovered": correct[~covered].float().mean().item()
                         if (~covered).any() else float("nan"),
        })
    unchanged = all(
        torch.equal(old, new) for old, new in zip(before, model.parameters())
    )
    return rows, unchanged


def run_context_seed(seed: int, steps: int = 1200) -> dict:
    torch.manual_seed(seed)
    generator = torch.Generator().manual_seed(seed)
    model = ContextLearner()
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-3, weight_decay=1e-2)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=steps, eta_min=3e-5
    )
    model.train()
    for step in range(steps):
        k = 1 + step % 4
        bank = sample_episode_bank(256, generator)
        tokens, targets, _ = render_episodes(bank, k)
        loss = F.cross_entropy(model(tokens), targets)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
    rows, unchanged = evaluate_context(model, seed)
    return {
        "seed": seed,
        "rows": rows,
        "unchanged": unchanged,
        "parameters": sum(parameter.numel() for parameter in model.parameters()),
    }


CONTEXT_SEEDS = list(range(6050, 6055))
context_runs = [run_context_seed(seed) for seed in CONTEXT_SEEDS]
context_accuracy = np.array([
    [row["accuracy"] for row in run["rows"]] for run in context_runs
])
context_ceiling = np.array([0.25, 0.50, 0.75, 1.00, 1.00])

print("model parameters:", f'{context_runs[0]["parameters"]:,}')
print("k  mean accuracy  seed sd  information ceiling")
# [4]
for k in range(5):
    print(
        f"{k}      {context_accuracy[:, k].mean():.3f}       "
        f"{context_accuracy[:, k].std(ddof=1):.3f}          "
        f"{context_ceiling[k]:.2f}"
    )
print("all evaluation passes left weights unchanged:",
      all(run["unchanged"] for run in context_runs))

**Plan**

1. Prepare the inputs and fixed settings for the example.
2. Define the reusable helpers: `make_rank_problem`, `FullUpdate`, and `LoRAUpdate`.
3. Define the reusable helpers: `train_linear_update` and `relative_update_error`.
4. Compare frozen, full, and low-rank updates against a planted rank-six correction.
5. Check the claimed identities, shapes, or invariants.
6. Report or visualize the measured result.

In [ ]:
# [1]
D_IN = D_OUT = 32
TARGET_RANK = 6
DIRECTION_MAGNITUDES = torch.tensor([1.20, 0.90, 0.65, 0.40, 0.22, 0.10])


# [2]
def make_rank_problem(
    seed: int = 1700,
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor,
           torch.Tensor, torch.Tensor]:
    generator = torch.Generator().manual_seed(seed)
    base = torch.randn(D_OUT, D_IN, generator=generator) / math.sqrt(D_IN)
    correction = torch.zeros(D_OUT, D_IN)
    diagonal = torch.arange(TARGET_RANK)
    correction[diagonal, diagonal] = DIRECTION_MAGNITUDES
    x_train = torch.randn(1024, D_IN, generator=generator)
    x_validation = torch.randn(4096, D_IN, generator=generator)
    y_train = F.linear(x_train, base + correction)
    y_validation = F.linear(x_validation, base + correction)
    return base, correction, x_train, y_train, x_validation, y_validation


class FullUpdate(nn.Module):
    def __init__(self, base: torch.Tensor) -> None:
        super().__init__()
        self.weight = nn.Parameter(base.clone())

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return F.linear(x, self.weight)


class LoRAUpdate(nn.Module):
    def __init__(
        self, base: torch.Tensor, rank: int, generator: torch.Generator
    ) -> None:
        super().__init__()
        self.register_buffer("weight0", base.clone())
        self.A = nn.Parameter(torch.randn(rank, D_IN, generator=generator) * 0.02)
        self.B = nn.Parameter(torch.zeros(D_OUT, rank))
        self.rank = rank
        self.alpha = rank

    @property
    def scaling(self) -> float:
        return self.alpha / self.rank

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        low_rank = F.linear(F.linear(x, self.A), self.B)
        return F.linear(x, self.weight0) + self.scaling * low_rank

    def merged_weight(self) -> torch.Tensor:
        return self.weight0 + self.scaling * (self.B @ self.A)


# [3]
def train_linear_update(
    model: nn.Module, x: torch.Tensor, y: torch.Tensor, steps: int = 1200
) -> None:
    optimizer = torch.optim.Adam(model.parameters(), lr=0.03)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=steps, eta_min=3e-4
    )
    for _ in range(steps):
        loss = F.mse_loss(model(x), y)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
        scheduler.step()


def relative_update_error(
    estimate: torch.Tensor, target: torch.Tensor
) -> float:
    return (torch.linalg.norm(estimate - target) / torch.linalg.norm(target)).item()


base, target_delta, x_train, y_train, x_val, y_val = make_rank_problem()
target_energy = torch.sum(DIRECTION_MAGNITUDES ** 2)

full_model = FullUpdate(base)
# [4]
train_linear_update(full_model, x_train, y_train)
with torch.no_grad():
    full_mse = F.mse_loss(full_model(x_val), y_val).item()
    full_relative_error = relative_update_error(
        full_model.weight - base, target_delta
    )
    frozen_mse = F.mse_loss(F.linear(x_val, base), y_val).item()

LORA_RANKS = [1, 2, 4, 6, 8]
lora_runs = []
# [5]
for seed in range(6050, 6055):
    for rank in LORA_RANKS:
        model = LoRAUpdate(
            base, rank, torch.Generator().manual_seed(seed + rank)
        )
        frozen_before = model.weight0.clone()
        train_linear_update(model, x_train, y_train)
        with torch.no_grad():
            merged = model.merged_weight()
            merge_error = (
                model(x_val[:64]) - F.linear(x_val[:64], merged)
            ).abs().max().item()
            assert torch.equal(frozen_before, model.weight0)
            assert merge_error < 2e-6
            learned_delta = merged - base
            lora_runs.append({
                "seed": seed,
                "rank": rank,
                "trainable": rank * (D_IN + D_OUT),
                "validation_mse": F.mse_loss(model(x_val), y_val).item(),
                "relative_error": relative_update_error(learned_delta, target_delta),
                "merge_error": merge_error,
            })

# [6]
print(f"frozen validation MSE: {frozen_mse:.6f}")
print(
    f"full update: MSE={full_mse:.2e}, relative delta error="
    f"{full_relative_error:.2e} (1,024 trainable)"
)
print("rank  trainable  mean validation MSE  mean relative update error")
for rank in LORA_RANKS:
    selected = [row for row in lora_runs if row["rank"] == rank]
    print(
        f"{rank:>4}  {selected[0]['trainable']:>9,}  "
        f"{np.mean([row['validation_mse'] for row in selected]):>19.8f}  "
        f"{np.mean([row['relative_error'] for row in selected]):>26.6f}"
    )
print("largest merged/unmerged difference:",
      f"{max(row['merge_error'] for row in lora_runs):.2e}")

**Plan**

1. Prepare the inputs and fixed settings for the example.
2. Define the reusable helpers: `make_quantization_problem`, `symmetric_quantize`, and `run_quantization_audit`.
3. Compare per-tensor and per-row symmetric quantization, including output error and metadata.

In [ ]:
# [1]
QUANT_D_IN, QUANT_D_OUT = 256, 64


# [2]
def make_quantization_problem(
    seed: int = 1701,
) -> tuple[torch.Tensor, torch.Tensor]:
    generator = torch.Generator().manual_seed(seed)
    directions = torch.randn(QUANT_D_OUT, QUANT_D_IN, generator=generator)
    directions = directions / directions.abs().amax(dim=1, keepdim=True)
    row_ranges = torch.logspace(-2, 1, QUANT_D_OUT).unsqueeze(1)
    weights = directions * row_ranges
    inputs = torch.randn(4096, QUANT_D_IN, generator=generator)
    return weights, inputs


def symmetric_quantize(
    weights: torch.Tensor, bits: int, per_row: bool
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    qmax = 2 ** (bits - 1) - 1
    scale = (
        weights.abs().amax(dim=1, keepdim=True) / qmax
        if per_row else weights.abs().amax() / qmax
    )
    scale = torch.where(scale == 0, torch.ones_like(scale), scale)
    codes = torch.clamp(torch.round(weights / scale), -qmax, qmax).to(torch.int8)
    reconstructed = codes.float() * scale
    return codes, reconstructed, scale


@torch.no_grad()
def run_quantization_audit() -> list[dict]:
    weights, inputs = make_quantization_problem()
    reference = inputs @ weights.T
    rows = []
    for bits in (8, 4):
        for per_row in (False, True):
            codes, reconstructed, scale = symmetric_quantize(
                weights, bits, per_row
            )
            row_errors = (
                torch.linalg.norm(reconstructed - weights, dim=1)
                / torch.linalg.norm(weights, dim=1)
            )
            payload_bytes = codes.numel() * bits / 8
            metadata_bytes = scale.numel() * 4
            rows.append({
                "bits": bits,
                "granularity": "per-row" if per_row else "per-tensor",
                "weight_error": (
                    torch.linalg.norm(reconstructed - weights)
                    / torch.linalg.norm(weights)
                ).item(),
                "output_error": (
                    torch.linalg.norm(inputs @ reconstructed.T - reference)
                    / torch.linalg.norm(reference)
                ).item(),
                "row_errors": row_errors.numpy(),
                "row_ranges": weights.abs().amax(dim=1).numpy(),
                "payload_bytes": int(payload_bytes),
                "metadata_bytes": int(metadata_bytes),
                "total_bytes": int(payload_bytes + metadata_bytes),
            })
    return rows


quant_rows = run_quantization_audit()
print("bits  scheme       w err   out err  med row  max row  payload  meta  total")
# [3]
for row in quant_rows:
    print(
        f"{row['bits']:>4}  {row['granularity']:<10}  "
        f"{row['weight_error']:.4f}      {row['output_error']:.4f}      "
        f"{np.median(row['row_errors']):.4f}      {row['row_errors'].max():.4f}  "
        f"{row['payload_bytes']:>7,}  {row['metadata_bytes']:>8,}  "
        f"{row['total_bytes']:>5,}"
    )